<h3 style="color:#6FA8DC; font-weight:bold">03 — Outlier Detection using IQR</h3>

This notebook covers the **Interquartile Range (IQR) method** from theory to complete working examples.

Topics:
- Q1, Q2, Q3
- IQR
- Lower and upper fences
- Manual calculation
- Pandas implementation
- Finding/removing outliers
- Capping using IQR
- Boxplot interpretation
- Comparing IQR with Z-Score
- Why IQR is useful for skewed data
- Train/test and production considerations

<h5 style="color:#78B89A; font-weight:bold;">What is IQR? → simple meaning</h5>

IQR means **Interquartile Range**.

```text
IQR = Q3 - Q1
```

- Q1 = 25th percentile
- Q2 = 50th percentile = median
- Q3 = 75th percentile

IQR represents the spread of the **middle 50%** of the data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

data = pd.Series([10, 12, 11, 13, 12, 14, 15, 13, 11, 12, 100])
df = pd.DataFrame({"value": data})

Q1 = df["value"].quantile(0.25)
Q3 = df["value"].quantile(0.75)
IQR = Q3 - Q1

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)

<h5 style="color:#78B89A; font-weight:bold;">IQR fences → the main formula</h5>

The common rule is:

```text
Lower Fence = Q1 - 1.5 × IQR
Upper Fence = Q3 + 1.5 × IQR
```

Values outside these fences are **potential outliers**.

In [ ]:
lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

print("Lower fence:", lower_fence)
print("Upper fence:", upper_fence)

df["is_outlier"] = (
    (df["value"] < lower_fence) |
    (df["value"] > upper_fence)
)

df

<h5 style="color:#78B89A; font-weight:bold;">Extract outliers</h5>

In [ ]:
outliers = df[df["is_outlier"]]
display(outliers)

<h5 style="color:#78B89A; font-weight:bold;">Remove outliers</h5>

In [ ]:
df_without_outliers = df[~df["is_outlier"]].copy()

print("Before:", len(df))
print("After:", len(df_without_outliers))

df_without_outliers

<h5 style="color:#78B89A; font-weight:bold;">IQR Capping → keep the rows</h5>

Instead of deleting observations, replace values outside the fences with the nearest fence.

In [ ]:
df["value_capped"] = df["value"].clip(
    lower=lower_fence,
    upper=upper_fence
)

df[["value", "value_capped"]]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(x=df["value"], ax=axes[0])
axes[0].set_title("Before IQR Capping")

sns.boxplot(x=df["value_capped"], ax=axes[1])
axes[1].set_title("After IQR Capping")

plt.tight_layout()
plt.show()

<h5 style="color:#78B89A; font-weight:bold;">IQR with a skewed dataset → why it is useful</h5>

The IQR method does not require the feature to be perfectly normally distributed.

This makes it a useful starting point for **skewed numerical data**.

In [ ]:
np.random.seed(42)

income = np.random.lognormal(mean=10.5, sigma=0.5, size=500)

income_df = pd.DataFrame({"income": income})

Q1 = income_df["income"].quantile(0.25)
Q3 = income_df["income"].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

income_df["is_outlier"] = (
    (income_df["income"] < lower) |
    (income_df["income"] > upper)
)

print("Potential outliers:", income_df["is_outlier"].sum())

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(x=income_df["income"])
plt.title("Skewed Data + IQR Detection")
plt.show()

<h5 style="color:#78B89A; font-weight:bold;">Create a reusable IQR function</h5>

This is useful when you need to inspect multiple numerical columns.

In [ ]:
def iqr_outlier_info(series, multiplier=1.5):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - multiplier * IQR
    upper = Q3 + multiplier * IQR

    mask = (series < lower) | (series > upper)

    return {
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "lower_fence": lower,
        "upper_fence": upper,
        "outlier_count": mask.sum()
    }

iqr_outlier_info(income_df["income"])

<h5 style="color:#78B89A; font-weight:bold;">IQR vs Z-Score</h5>

| IQR | Z-Score |
|---|---|
| Uses Q1 and Q3 | Uses mean and standard deviation |
| More robust to extreme values | Sensitive to extreme values |
| Good for skewed data | Better for approximately normal data |
| Uses 1.5 × IQR rule | Common rule uses `|Z| > 3` |

<h5 style="color:#78B89A; font-weight:bold;">Production consideration</h5>

For ML preprocessing:

```text
Train data
   ↓
Calculate Q1/Q3/IQR
   ↓
Save fences
   ↓
Apply same fences to test/new data
```

Do not calculate separate IQR limits on every production batch if the intention is to reproduce the training preprocessing logic.

<h3 style="color:#6FA8DC; font-weight:bold">IQR Revision</h3>

```text
Q1 = 25th percentile
Q3 = 75th percentile

IQR = Q3 - Q1

Lower = Q1 - 1.5 × IQR
Upper = Q3 + 1.5 × IQR

Outside limits
      ↓
Potential outlier
      ↓
Investigate → remove / keep / cap / transform
```

⭐ IQR is one of the most useful general-purpose methods for **univariate numerical outlier detection**.